## SITCOM-2062: Rubin thermal data retrieval
### Karla Peña Ramírez
May 7 2025

Here you can find the proposed code for data retrieval of either ComCam or LSSTCam temperature sensor data.

In [ ]:
#Setting packages
import asyncio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import sys
import time
import warnings
import seaborn as sns
import datetime as dt
from scipy import stats
from matplotlib.dates import DateFormatter
from astropy.table import Table, join
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.ensemble import IsolationForest
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import cross_val_score
from scipy.cluster.hierarchy import dendrogram, linkage
from lsst.utils.packages import getEnvironmentPackages
from IPython.display import Markdown, display
#%matplotlib widget

from astropy.time import Time, TimeDelta
from lsst_efd_client.efd_helper import merge_packed_time_series
from lsst.meas.algorithms.installGaussianPsf import FwhmPerSigma
from lsst.daf.butler import Butler 
#from tqdm.notebook import tqdm


from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import Ridge
from sklearn.model_selection import ShuffleSplit
import warnings

# Ignore the many warning messages from ``merge_packed_time_series``
warnings.simplefilter(action="ignore", category=FutureWarning)

#-------Clients
from lsst_efd_client import EfdClient
client = EfdClient("usdf_efd")

#from lsst.summit.utils.efdUtils import makeEfdClient
#client = makeEfdClient()

#from lsst.summit.utils import ConsDbClient
#os.environ["no_proxy"] += ",.consdb"
#url="http://consdb-pq.consdb:8080/consdb"
#consdb=ConsDbClient(url)
#-------Clients

def print_session_info():
    # Time info
    print(f"# Session Info on {time.strftime('%Y-%m-%d at %H:%M:%S %Z', time.localtime(time.time()))}\n")

    # Python info
    print(f"## Python Interpreter\n\nVersion: {sys.version}  \nExecutable: {sys.executable}\n")

    # LSST info
    packages = getEnvironmentPackages(True)
    dev_packages = {"lsst_distrib": packages["lsst_distrib"]}
    dev_packages.update({k: v.split("@")[0] for k, v in packages.items() if "LOCAL" in v})
    print("## Science Pipelines\n\n" + "\n".join(f"{k:<20} {v}" for k, v in dev_packages.items()))

## 1. Data retrievals. 
Use either ComCam or LSSTCam on-sky temperature sensor data. We are going to use the image time stamps of the acquired images as a proxy of on-sky dome aperture.
### On-sky time ranges:

In [ ]:
#Identify the repo where all the data lives and the latest consolidated processing collection
#Run > butler query-collections /repo/main "*DRP*w_2025_0*" | grep CHAINED
######ComCam
repo = '/repo/main'
instrument = "LSSTComCam"
collection = 'LSSTComCam/runs/DRP/DP1/w_2025_08/DM-49029'
#####LSSTCam
repo = '/repo/embargo'
instrument = "LSSTCam"
collection = 'LSSTCam/raw/all','LSSTCam/runs/nightlyValidation'

butler = Butler(repo, collections=collection)
registry = butler.registry


In [ ]:
#Query the metadata for the **`exposure`** dimension, limiting the results to this particular instrument and range of days of observation:
######ComCam
instrument = 'LSSTComCam'
day_obs_start = 20241017
day_obs_end = 20241212
#####LSSTCam
instrument = 'LSSTCam'
day_obs_start = 20250415
day_obs_end = 20250427

query="instrument='%s' AND day_obs>=%d AND day_obs<=%d" % (instrument, day_obs_start, day_obs_end)
results = registry.queryDimensionRecords('exposure', where=query)

In [ ]:
#Taken from vv-team-notebooks/reports/TargetReport.ipynb
#Stop executing if there are no results returned:
n_results = results.count()

if n_results <= 0:
    raise StopExecution
else:
    print("""There are %d results returned from querying the butler for instrument %s between dates %d and %d (inclusive).""" % 
          (n_results, instrument, day_obs_start, day_obs_end))

In [ ]:
#Taken from vv-team-notebooks/reports/TargetReport.ipynb
#Instantiate a pandas `DataFrame` with useful columns available in the `exposure` dimension:
df_exp = pd.DataFrame(columns=['id', 'obs_id','day_obs', 'seq_num',
                                    'time_start','time_end' ,'type', 'reason', 
                                    'target','filter','zenith_angle',
                                    'expos','ra','dec','skyangle',
                                    'azimuth','zenith','science_program',
                                    'jd','mjd'])

In [ ]:
#Taken from vv-team-notebooks/reports/TargetReport.ipynb
#Read the query results into the new pandas `DataFrame`:
for count, info in enumerate(results):
    
    try:

        df_exp.loc[count] = [info.id, info.obs_id, info.day_obs, info.seq_num, 
                                  info.timespan.begin.utc.iso,
                                  info.timespan.end.utc.iso, 
                                  info.observation_type, info.observation_reason, info.target_name, 
                                  info.physical_filter, info.zenith_angle, 
                                  info.exposure_time,info.tracking_ra, info.tracking_dec, 
                                  info.sky_angle,info.azimuth ,info.zenith_angle, 
                                  info.science_program, info.timespan.begin.jd, info.timespan.begin.mjd]

    except:
    
        print(">>>   Unexpected error:", sys.exc_info()[0])
        info_timespan_begin_to_string = "2021-01-01 00:00:00.00"
        info_timespan_end_to_string = "2051-01-01 00:00:00.00"
        info_timespan_begin_jd = 0
        info_timespan_begin_mjd = 0
        df_exp.loc[count] = [info.id, info.obs_id, info.day_obs, info.seq_num, 
                                  pd.to_datetime(info_timespan_begin_to_string),
                                  pd.to_datetime(info_timespan_end_to_string), 
                                  info.observation_type, info.observation_reason, info.target_name, 
                                  info.physical_filter, info.zenith_angle, 
                                  info.exposure_time,info.tracking_ra, info.tracking_dec, 
                                  info.sky_angle,info.azimuth ,info.zenith_angle, 
                                  info.science_program, info_timespan_begin_jd, info_timespan_begin_mjd ]    

In [ ]:
#Taken from vv-team-notebooks/reports/TargetReport.ipynb
#Clean-up the dataframe
#Re-cast the `id`, `day_obs`, and `seq_num` rows as `int`'s:
df_exp = df_exp.astype({"id": int,'day_obs': int,'seq_num':int})
# Replace `NaN`'s in the `ra` and `dec` columns with zero.  
df_exp['ra'] = df_exp['ra'].fillna(0)
df_exp['dec'] = df_exp['dec'].fillna(0)
#Select only on-sky data
df_exp.type.unique()
df_open = df_exp[(df_exp.type == 'science') | (df_exp.type == 'cwfs') | (df_exp.type == 'focus') | (df_exp.type == 'acq')| (df_exp.type == 'flat')]

In [ ]:
#Group data by dayobs identifying the extremme dates for the exposures.
result_on_sky = df_open.groupby('day_obs').agg({
    'time_start': 'min',
    'time_end': 'max'
}).reset_index()

### Telemetry retrieval:

In [ ]:
def read_time_data():
    df = result_on_sky
    df['time_start'] = pd.to_datetime(df['time_start'])
    df['time_end'] = pd.to_datetime(df['time_end'])
    return df
    
def get_time_range(df, day_obs):
    day_data = df[df['day_obs'] == day_obs].iloc[0]
    return day_data['time_start'], day_data['time_end']

In [ ]:
async def get_dynalene_data(client, day_obs, sampling="1h"):
    """Get dynalene data for a specific day_obs."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(dynTMAsupTS01) AS TMA_Supply_Dynalene, 
        mean(dynTMAretTS02) AS TMA_Return_Dynalene, 
        mean(dynCH01supTS05) AS TMA_Chiller_1
    FROM "lsst.sal.HVAC.dynaleneP05" 
    WHERE time > '{start_time.isoformat()}Z' 
    AND time < '{end_time.isoformat()}Z' 
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


async def get_dynalene_mtmount(client, day_obs, sampling="1h"):
    """Get MTMount dynalene data for a specific day_obs."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(dynaleneTemperatureAzimuth0001) AS TMA_Azimuth_1,
        mean(dynaleneTemperatureAzimuth0002) AS TMA_Azimuth_2,
        mean(dynaleneTemperaturePier0102) AS TMA_Pier_2
    FROM "lsst.sal.MTMount.dynaleneCooling"
    WHERE time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


async def get_ess_1(client, day_obs, sampling="1h"):
    """Get ESS temperature data for CamHex and CamRot."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(temperatureItem0) AS CamHex_Strut_7,
        mean(temperatureItem1) AS CamHex_Strut_8,
        mean(temperatureItem2) AS CamHex_Strut_9,
        mean(temperatureItem3) AS CamHex_Strut_10,
        mean(temperatureItem4) AS CamHex_Strut_11,
        mean(temperatureItem5) AS CamHex_Strut_12,
        mean(temperatureItem6) AS CamRot_Motor_1,
        mean(temperatureItem7) AS CamRot_Motor_2
    FROM "lsst.sal.ESS.temperature"
    WHERE salIndex = 1
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


async def get_ess_106(client, day_obs, sampling="1h"):
    """Get ESS temperature data for M2 Tangent Links."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(temperatureItem0) AS M2_Tangent_Link_A1,
        mean(temperatureItem1) AS M2_Tangent_Link_A2,
        mean(temperatureItem2) AS M2_Tangent_Link_A3,
        mean(temperatureItem3) AS M2_Tangent_Link_A4,
        mean(temperatureItem4) AS M2_Tangent_Link_A5,
        mean(temperatureItem5) AS M2_Tangent_Link_A6
    FROM "lsst.sal.ESS.temperature"
    WHERE salIndex = 106
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)

async def get_ess_112(client, day_obs, sampling="1h"):
    """Get ESS temperature data for M2."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(temperatureItem0) AS ESS_112_M2
    FROM "lsst.sal.ESS.temperature"
    WHERE salIndex = 112
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


async def get_ess_111(client, day_obs, sampling="1h"):
    """Get ESS temperature data for Camera."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(temperatureItem0) AS ESS_111_Camera
    FROM "lsst.sal.ESS.temperature"
    WHERE salIndex = 111
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


async def get_ess_113(client, day_obs, sampling="1h"):
    """Get ESS temperature data for Dome."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(temperatureItem0) AS ESS_113_Dome
    FROM "lsst.sal.ESS.temperature"
    WHERE salIndex = 113
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


async def get_ess_301(client, day_obs, sampling="1h"):
    """Get ESS temperature data for Outside Dome."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(temperatureItem0) AS Outside_Dome
    FROM "lsst.sal.ESS.temperature"
    WHERE salIndex = 301
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


async def get_mtmount_topend_chiller(client, day_obs, sampling="1h"):
    """Get MTMount TopEndChiller temperature data."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(temperatureSensor0501) AS Temperature_0501,
        mean(ambientTemperatureSensor0502) AS Ambient_0502,
        mean(ductTemperatureSensor0506) AS Duct_0506,
        mean(ductTemperatureSensor0507) AS Duct_0507,
        mean(internalTemperatureElectricalCabinet0) AS Internal_Cabinet_0,
        mean(internalTemperatureElectricalCabinet1) AS Internal_Cabinet_1,
        mean(externalTemperatureElectricalCabinet1) AS External_Cabinet_1,
        mean(internalTemperatureElectricalCabinet3) AS Internal_Cabinet_3,
        mean(externalTemperatureElectricalCabinet3) AS External_Cabinet_3
    FROM "lsst.sal.MTMount.topEndChiller"
    WHERE time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


async def get_m2_temps(client, day_obs, sampling="1h"):
    """Get M2 temperature data."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(intake0) as M2_Intake0,
        mean(intake1) as M2_Intake1,
        mean(exhaust0) as M2_Exhaust0,
        mean(exhaust1) as M2_Exhaust1,
        mean(ring5) as M2_ring
    FROM "lsst.sal.MTM2.temperature"
    WHERE time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


async def get_m1m3_glycol(client, day_obs, sampling="1h"):
    """Get M1M3 glycol temperature data."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean(telescopeCoolantReturnTemperature) as TMA_Coolant_Retun,
        mean(telescopeCoolantSupplyTemperature) as TMA_Coolant_Supply,
        mean(mirrorCoolantSupplyTemperature) as M1M3_Coolant_Supply,
        mean(mirrorCoolantReturnTemperature) as M1M3_Coolant_Return,
        mean(insideCellTemperature1) as Interior_Cell_1,
        mean(insideCellTemperature2) as Interior_Cell_2,
        mean(insideCellTemperature3) as Interior_Cell_3
    FROM "lsst.sal.MTM1M3TS.glycolLoopTemperature"
    WHERE time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


async def get_m1m3_fcu(client, day_obs, sampling="1h"):
    """Get M1M3 FCU temperature data."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    
    query = f"""
    SELECT 
        mean("absoluteTemperature20") AS "FCU20",
        mean("absoluteTemperature50") AS "FCU50",
        mean("absoluteTemperature80") AS "FCU80"
    FROM "lsst.sal.MTM1M3TS.thermalData"
    WHERE time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)    


async def get_glycolMTMount(client, day_obs, sampling="1h"):
    """Get MTMount glycol general purpose temperature data."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    query = f"""
    SELECT 
        mean(glycolTemperaturePier0001) AS General_Glycol_L6_1,
        mean(glycolTemperaturePier0002) AS General_Glycol_L6_2
    FROM "lsst.sal.MTMount.generalPurposeGlycolWater" 
    WHERE time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)    


async def get_coldglycolMTMount(client, day_obs, sampling="1h"):
    """Get MTMount cold glycol temperature data."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
    query = f"""
    SELECT 
        mean(glycolTemperaturePier0101) AS Cold_Glycol_L6
    FROM "lsst.sal.MTMount.cooling" 
    WHERE time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)    


### Usage:

In [ ]:
#Single day data:
sampling = '1m'
day_to_check = '202nnnnn' #Example: 20250430

#Dynalene data
dynalene = await get_dynalene_data(client, sampling = sampling, day_obs=day_to_check)
dynaleneMTMount = await get_dynalene_mtmount(client, sampling = sampling, day_obs=day_to_check)
#ESS temperature data
ESS_1 = await get_ess_1(client, sampling = sampling, day_obs=day_to_check)  # CamHex and CamRot
ESS_106 = await get_ess_106(client, sampling = sampling, day_obs=day_to_check)  # M2 Tangent Links
ESS_112 = await get_ess_112(client, sampling = sampling, day_obs=day_to_check)  # M2
ESS_111 = await get_ess_111(client, sampling = sampling, day_obs=day_to_check)  # Camera
ESS_113 = await get_ess_113(client, sampling = sampling, day_obs=day_to_check)  # Dome
ESS_301 = await get_ess_301(client, sampling = sampling, day_obs=day_to_check)  # Outside Dome
#MTMount TopEndChiller data
MTMountTopEndChiller = await get_mtmount_topend_chiller(client, sampling = sampling, day_obs=day_to_check)
#M2 temperature data
M2Temps = await get_m2_temps(client, sampling = sampling, day_obs=day_to_check)
#M1M3 data
M1M3_glycol = await get_m1m3_glycol(client, sampling = sampling, day_obs=day_to_check)  # Glycol temperatures
M1M3_FCU = await get_m1m3_fcu(client, sampling = sampling, day_obs=day_to_check)  # FCU temperatures
mask1 = M1M3_FCU['FCU20'] <= 100
mask2 = M1M3_FCU['FCU50'] <= 100
mask3 = M1M3_FCU['FCU80'] <= 100
M1M3_FCU_red1 = M1M3_FCU[mask1]
M1M3_FCU_red2 = M1M3_FCU[mask2]
M1M3_FCU_red3 = M1M3_FCU[mask3]
#Glycol
glycolMTMount = await get_glycolMTMount(client, sampling = sampling, day_obs=day_to_check)
coldglycolMTMount = await get_coldglycolMTMount(client, sampling = sampling, day_obs=day_to_check)

#Single day
data_objects = [
M1M3_glycol['M1M3_Coolant_Return'],
M1M3_FCU_red1['FCU20'],
M1M3_FCU_red2['FCU50'],
M1M3_FCU_red3['FCU80'],
M1M3_glycol['Interior_Cell_3'],
M1M3_glycol['Interior_Cell_1'],
M1M3_glycol['Interior_Cell_2'],
ESS_301['Outside_Dome'],
ESS_106['M2_Tangent_Link_A6'],
ESS_106['M2_Tangent_Link_A2'],
ESS_106['M2_Tangent_Link_A4'],
MTMountTopEndChiller['External_Cabinet_3'],
MTMountTopEndChiller['Duct_0507'],
MTMountTopEndChiller['Ambient_0502'],
MTMountTopEndChiller['Duct_0506'],
M2Temps['M2_Exhaust0'],
M2Temps['M2_Intake1'],
M2Temps['M2_Exhaust1'],
M2Temps['M2_Intake0'],
MTMountTopEndChiller['Temperature_0501'],
ESS_113['ESS_113_Dome'],
MTMountTopEndChiller['External_Cabinet_1'],
ESS_112['ESS_112_M2'],
ESS_111['ESS_111_Camera'],
ESS_106['M2_Tangent_Link_A5'],
ESS_106['M2_Tangent_Link_A3'],
M2Temps['M2_ring'],
ESS_1['CamRot_Motor_1'],    
ESS_1['CamRot_Motor_2'],
ESS_1['CamHex_Strut_11'],
ESS_1['CamHex_Strut_7'],
ESS_1['CamHex_Strut_8'],
ESS_1['CamHex_Strut_12'],
ESS_1['CamHex_Strut_9'],
ESS_1['CamHex_Strut_10'],
M1M3_glycol['TMA_Coolant_Supply'],
M1M3_glycol['TMA_Coolant_Retun'],
M1M3_glycol['M1M3_Coolant_Supply'],
MTMountTopEndChiller['Internal_Cabinet_0'],
MTMountTopEndChiller['Internal_Cabinet_1'],
MTMountTopEndChiller['Internal_Cabinet_3'],
dynaleneMTMount['TMA_Pier_2'],
dynalene['TMA_Return_Dynalene'],
dynaleneMTMount['TMA_Azimuth_2'],
dynalene['TMA_Chiller_1'],
dynaleneMTMount['TMA_Azimuth_1'],
dynalene['TMA_Supply_Dynalene'],
glycolMTMount['General_Glycol_L6_1'],
glycolMTMount['General_Glycol_L6_2'],  
coldglycolMTMount['Cold_Glycol_L6']
]

df = pd.concat(data_objects, axis=1)

In [ ]:
#Reproducibility
print_session_info()